In [1]:
import pandas as pd

df = pd.read_csv("world_health_data.csv")

# Display shape and data types
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)

Shape: (6650, 12)

Data types:
country                   object
country_code              object
year                       int64
health_exp               float64
life_expect              float64
maternal_mortality       float64
infant_mortality         float64
neonatal_mortality       float64
under_5_mortality        float64
prev_hiv                 float64
inci_tuberc              float64
prev_undernourishment    float64
dtype: object


In [2]:
# Count and percentage of missing values per column
missing_count = df.isnull().sum()
missing_pct   = (missing_count / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "Missing Count"  : missing_count,
    "Missing (%)"    : missing_pct
})

# Show only columns that have at least one missing value
print(missing_df[missing_df["Missing Count"] > 0].sort_values("Missing (%)", ascending=False))

                       Missing Count  Missing (%)
prev_hiv                        2380        35.79
prev_undernourishment           1845        27.74
maternal_mortality              1757        26.42
health_exp                      1483        22.30
inci_tuberc                     1221        18.36
infant_mortality                 794        11.94
neonatal_mortality               794        11.94
under_5_mortality                794        11.94
life_expect                      460         6.92


In [3]:
# Select only numeric columns excluding identifiers
numeric_cols = df.select_dtypes(include="number").columns.tolist()
numeric_cols = [c for c in numeric_cols if c != "year"]

outlier_summary = []

for col in numeric_cols:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR  # lower bound
    upper = Q3 + 1.5 * IQR  # upper bound

    n_outliers = df[(df[col] < lower) | (df[col] > upper)][col].count()

    outlier_summary.append({
        "Column"    : col,
        "Q1"        : round(Q1, 2),
        "Q3"        : round(Q3, 2),
        "IQR"       : round(IQR, 2),
        "Lower"     : round(lower, 2),
        "Upper"     : round(upper, 2),
        "Outliers"  : n_outliers
    })

outlier_df = pd.DataFrame(outlier_summary)
print(outlier_df.sort_values("Outliers", ascending=False))

                  Column     Q1      Q3     IQR   Lower   Upper  Outliers
6               prev_hiv   0.10    1.50    1.40   -2.00    3.60       554
7            inci_tuberc  14.00  177.00  163.00 -230.50  421.50       377
2     maternal_mortality  18.00  297.00  279.00 -400.50  715.50       260
5      under_5_mortality   9.90   60.10   50.20  -65.40  135.40       229
8  prev_undernourishment   2.63   15.97   13.33  -17.37   35.97       169
0             health_exp   4.25    7.79    3.54   -1.07   13.11       107
3       infant_mortality   8.40   44.40   36.00  -45.60   98.40        85
1            life_expect  64.44   76.61   12.17   46.18   94.87        50
4     neonatal_mortality   5.20   25.70   20.50  -25.55   56.45         9


In [4]:
# ── CORRECTION 1 : Missing values ─────────────────────────────────────────────

# Impute missing values with median per country
df[numeric_cols] = df.groupby("country")[numeric_cols].transform(
    lambda x: x.fillna(x.median())
)

# Fallback : impute remaining missing values with global median
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Verify no missing values remain
print("Missing values after correction:")
print(df[numeric_cols].isnull().sum())


# ── CORRECTION 2 : Outliers (capping IQR) ─────────────────────────────────────

# Cap outliers at IQR bounds to limit distortion in aggregations
# We preserve the values — we only cap statistical extremes, not geopolitical realities

for col in numeric_cols:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    # Cap values below lower bound and above upper bound
    df[col] = df[col].clip(lower=lower, upper=upper)

# Confirm correction applied
print("\nOutlier capping applied on:", numeric_cols)
print(df[numeric_cols].describe().round(2))

Missing values after correction:
health_exp               0
life_expect              0
maternal_mortality       0
infant_mortality         0
neonatal_mortality       0
under_5_mortality        0
prev_hiv                 0
inci_tuberc              0
prev_undernourishment    0
dtype: int64

Outlier capping applied on: ['health_exp', 'life_expect', 'maternal_mortality', 'infant_mortality', 'neonatal_mortality', 'under_5_mortality', 'prev_hiv', 'inci_tuberc', 'prev_undernourishment']
       health_exp  life_expect  maternal_mortality  infant_mortality  \
count     6650.00      6650.00             6650.00           6650.00   
mean         6.02        70.19              163.71             27.84   
std          2.36         8.49              192.84             23.44   
min          1.11        47.37                1.00              1.30   
25%          4.41        64.84               22.00              9.40   
50%          5.41        71.85               74.00             20.10   
75%        

In [5]:
# ── CONVERSION 1 : year → datetime (compatible Power BI time axis) ─────────────

# Convert year integer to datetime for time-series visuals in Power BI
df["year"] = pd.to_datetime(df["year"], format="%Y")


# ── CONVERSION 2 : float columns → rounded to 2 decimal places ────────────────

# List of all numeric indicator columns
float_cols = [
    "health_exp",
    "life_expect",
    "maternal_mortality",
    "infant_mortality",
    "neonatal_mortality",
    "under_5_mortality",
    "prev_hiv",
    "inci_tuberc",
    "prev_undernourishment"
]

# Round to 2 decimals for SQL DECIMAL(10,2) compatibility and clean Power BI display
df[float_cols] = df[float_cols].round(2)


# ── CONVERSION 3 : country, country_code → string (VARCHAR) ───────────────────

# Ensure text columns are clean strings, no mixed types
df["country"]      = df["country"].astype(str).str.strip()
df["country_code"] = df["country_code"].astype(str).str.strip()


# ── VERIFICATION ───────────────────────────────────────────────────────────────

# Confirm all types after conversion
print("Data types after conversion:")
print(df.dtypes)
print("\nSample after conversion:")
print(df.head(3))

Data types after conversion:
country                          object
country_code                     object
year                     datetime64[ns]
health_exp                      float64
life_expect                     float64
maternal_mortality              float64
infant_mortality                float64
neonatal_mortality              float64
under_5_mortality               float64
prev_hiv                        float64
inci_tuberc                     float64
prev_undernourishment           float64
dtype: object

Sample after conversion:
                       country country_code       year  health_exp  \
0                        Aruba          ABW 1999-01-01        5.41   
1  Africa Eastern and Southern          AFE 1999-01-01        5.97   
2                  Afghanistan          AFG 1999-01-01        9.93   

   life_expect  maternal_mortality  infant_mortality  neonatal_mortality  \
0        73.56               74.00              20.1               12.61   
1        51.26    

In [6]:
# ── RENOMMAGE MÉTIER : toutes les variables ambiguës ou abrégées ───────────────

# Rename columns to business-friendly names readable in Power BI slicers and KPI cards
df = df.rename(columns={
    "health_exp"             : "health_spending_pct_gdp",      # spending % of GDP
    "life_expect"            : "life_expectancy_years",         # unit embedded in name
    "maternal_mortality"     : "maternal_mortality_rate",       # adds 'rate' for clarity
    "infant_mortality"       : "infant_mortality_rate",         # distinguishes from neonatal
    "neonatal_mortality"     : "neonatal_mortality_rate",       # medical term kept + rate
    "under_5_mortality"      : "under5_mortality_rate",         # snake_case, clear target
    "prev_hiv"               : "hiv_prevalence_pct",            # removes ambiguous 'prev'
    "inci_tuberc"            : "tuberculosis_incidence_rate",   # no abbreviation
    "prev_undernourishment"  : "undernourishment_prevalence_pct" # unit % embedded
})


# ── VERIFICATION ───────────────────────────────────────────────────────────────

# Confirm all column names after renaming
print("Columns after renaming:")
print(df.columns.tolist())

print("\nSample after renaming:")
print(df.head(3))

Columns after renaming:
['country', 'country_code', 'year', 'health_spending_pct_gdp', 'life_expectancy_years', 'maternal_mortality_rate', 'infant_mortality_rate', 'neonatal_mortality_rate', 'under5_mortality_rate', 'hiv_prevalence_pct', 'tuberculosis_incidence_rate', 'undernourishment_prevalence_pct']

Sample after renaming:
                       country country_code       year  \
0                        Aruba          ABW 1999-01-01   
1  Africa Eastern and Southern          AFE 1999-01-01   
2                  Afghanistan          AFG 1999-01-01   

   health_spending_pct_gdp  life_expectancy_years  maternal_mortality_rate  \
0                     5.41                  73.56                    74.00   
1                     5.97                  51.26                   507.00   
2                     9.93                  54.85                   578.88   

   infant_mortality_rate  neonatal_mortality_rate  under5_mortality_rate  \
0                   20.1                    12.61 

In [7]:
from sklearn.preprocessing import MinMaxScaler

# ── FEATURE 1 : health_spending_category ──────────────────────────────────────

# Categorize health spending based on WHO thresholds (<5% Low, 5-9% Medium, ≥10% High)
df["health_spending_category"] = pd.cut(
    df["health_spending_pct_gdp"],
    bins    = [0, 5, 10, float("inf")],
    labels  = ["Low", "Medium", "High"],
    right   = False
)


# ── FEATURE 2 : avoidable_mortality_index ─────────────────────────────────────

# Normalize the 3 mortality indicators between 0 and 1
scaler = MinMaxScaler()

mortality_cols = [
    "infant_mortality_rate",
    "maternal_mortality_rate",
    "under5_mortality_rate"
]

# Normalize each mortality column independently
df[mortality_cols] = scaler.fit_transform(df[mortality_cols])

# Average normalized scores → synthetic avoidable mortality index (0=best, 1=worst)
df["avoidable_mortality_index"] = df[mortality_cols].mean(axis=1).round(3)


# ── FEATURE 3 : infectious_disease_index ──────────────────────────────────────

# Normalize the 3 infectious disease indicators between 0 and 1
infectious_cols = [
    "hiv_prevalence_pct",
    "tuberculosis_incidence_rate",
    "undernourishment_prevalence_pct"
]

# Normalize each infectious disease column independently
df[infectious_cols] = scaler.fit_transform(df[infectious_cols])

# Average normalized scores → synthetic infectious disease pressure index
df["infectious_disease_index"] = df[infectious_cols].mean(axis=1).round(3)


# ── FEATURE 4 : health_roi_score ──────────────────────────────────────────────

# ROI score = life expectancy gained per unit of health spending
# Avoid division by zero with replace(0, None)
df["health_roi_score"] = (
    df["life_expectancy_years"] / df["health_spending_pct_gdp"].replace(0, None)
).round(2)


# ── FEATURE 5 : life_expectancy_category ──────────────────────────────────────

# Categorize life expectancy into 4 decision-readable levels
df["life_expectancy_category"] = pd.cut(
    df["life_expectancy_years"],
    bins    = [0, 60, 70, 75, float("inf")],
    labels  = ["Critical", "Low", "Medium", "High"],
    right   = False
)


# ── FEATURE 6 : performance_gap_flag ──────────────────────────────────────────

# Flag countries with high spending but low/critical life expectancy (critical gap)
df["performance_gap_flag"] = (
    (df["health_spending_category"] == "High") &
    (df["life_expectancy_category"].isin(["Critical", "Low"]))
).map({True: "Critical Gap", False: "Normal"})


# ── VERIFICATION ───────────────────────────────────────────────────────────────

# Display new features overview
new_features = [
    "health_spending_category",
    "avoidable_mortality_index",
    "infectious_disease_index",
    "health_roi_score",
    "life_expectancy_category",
    "performance_gap_flag"
]

print("New features created:")
print(df[new_features].head(10))
print("\nData types:")
print(df[new_features].dtypes)
print("\nFinal dataset shape:", df.shape)

New features created:
  health_spending_category  avoidable_mortality_index  \
0                   Medium                      0.176   
1                   Medium                      0.959   
2                   Medium                      1.000   
3                      Low                      1.000   
4                      Low                      0.878   
5                   Medium                      0.172   
6                   Medium                      0.081   
7                      Low                      0.434   
8                      Low                      0.059   
9                   Medium                      0.147   

   infectious_disease_index  health_roi_score life_expectancy_category  \
0                     0.131             13.60                   Medium   
1                     0.672              8.59                 Critical   
2                     0.472              5.52                 Critical   
3                     0.237             13.23         

In [8]:
# Drop columns absorbed into synthetic indices or irrelevant for managers dashboard
cols_to_drop = [
    "neonatal_mortality_rate",        # not integrated in any final index
    "maternal_mortality_rate",        # captured by avoidable_mortality_index
    "infant_mortality_rate",          # captured by avoidable_mortality_index
    "under5_mortality_rate",          # captured by avoidable_mortality_index
    "hiv_prevalence_pct",             # captured by infectious_disease_index
    "tuberculosis_incidence_rate",    # captured by infectious_disease_index
    "undernourishment_prevalence_pct" # captured by infectious_disease_index
]

df = df.drop(columns=cols_to_drop)


# ── VERIFICATION ───────────────────────────────────────────────────────────────

# Confirm final columns retained
print("Final columns:", df.columns.tolist())
print("Final shape:", df.shape)
print("\nFinal dataset preview:")
print(df.head(5))

Final columns: ['country', 'country_code', 'year', 'health_spending_pct_gdp', 'life_expectancy_years', 'health_spending_category', 'avoidable_mortality_index', 'infectious_disease_index', 'health_roi_score', 'life_expectancy_category', 'performance_gap_flag']
Final shape: (6650, 11)

Final dataset preview:
                       country country_code       year  \
0                        Aruba          ABW 1999-01-01   
1  Africa Eastern and Southern          AFE 1999-01-01   
2                  Afghanistan          AFG 1999-01-01   
3   Africa Western and Central          AFW 1999-01-01   
4                       Angola          AGO 1999-01-01   

   health_spending_pct_gdp  life_expectancy_years health_spending_category  \
0                     5.41                  73.56                   Medium   
1                     5.97                  51.26                   Medium   
2                     9.93                  54.85                   Medium   
3                     3.76     

In [9]:
df.to_excel("world_health_data.xlsx",index=False,sheet_name="world_health")

In [10]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

user = "root"
password = "mysql2026"
host = "localhost"
port = 3306
db = "esmel"

engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:{port}/{db}",
    echo=False
)

In [11]:
engine.connect()

In [12]:
df.to_sql(
    name="world_health",
    con=engine,
    if_exists="replace",
    index=False
)

6650